In [1]:
import cv2
import numpy as np
import os

In [5]:
# Height Width
chessboardSize = (10, 13)
# Image size
frameSize = (640, 480)

# termination criteria
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Prepare Object Points Eg (0,0,0) (1,0,0)
objp = np.zeros((chessboardSize[0]*chessboardSize[1], 3), np.float32)
objp[:,:2] = np.mgrid[0:chessboardSize[0], 0:chessboardSize[1]].T.reshape(-1,2)

size_of_chessboard_squares_mm = 16
objp = objp * size_of_chessboard_squares_mm

# Arrays to Store Object Points & Image Points from all images
objPoints = [] # 3d point in real world space
imgPoints = [] # 2d points in image plane

all_images = 'myImages'
images = os.listdir(all_images)

for image in images:
    print(f'{all_images}/{image}')
    img = cv2.imread(f'{all_images}/{image}')
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Find Chessboard Corners
    ret, corners = cv2.findChessboardCorners(gray, chessboardSize, None)
    
    if ret == True:
        
        objPoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgPoints.append(corners)
        
        # Draw and display the corners
        cv2.drawChessboardCorners(img, chessboardSize, corners2, ret)
        cv2.imshow('Image', img)
        cv2.waitKey(0)
        
cv2.destroyAllWindows()

myImages/0.png
myImages/1.png
myImages/2.png
myImages/3.png
myImages/5.png
myImages/7.png


In [6]:
ret, cameraMatrix, dist, rvecs, tvecs = cv2.calibrateCamera(objPoints, imgPoints, frameSize, None, None)

print('Camera Calibrated: ', ret)
print('\nCamera Matrix:\n', cameraMatrix)
print('\nDistortion Parameters:\n', dist)
print('\nRotation Vectors:\n', rvecs)
print('\nTransalation Vectors:\n', tvecs)

with open('intrinsic1.npy', 'wb') as f:
    np.save(f, cameraMatrix)

Camera Calibrated:  0.6555267596847204

Camera Matrix:
 [[580.18421199   0.         306.32847071]
 [  0.         581.93758236 241.22427157]
 [  0.           0.           1.        ]]

Distortion Parameters:
 [[-3.30325200e-01  1.96909151e+00 -7.17910236e-03 -5.94455010e-04
  -4.54516181e+00]]

Rotation Vectors:
 (array([[ 0.01548053],
       [-0.12588815],
       [-1.5459756 ]]), array([[-0.04662464],
       [-0.327517  ],
       [-1.54230047]]), array([[-0.02040707],
       [-0.2571414 ],
       [-1.52476003]]), array([[-0.12224138],
       [-0.18382849],
       [-1.53319001]]), array([[-0.02809608],
       [-0.11720397],
       [-1.52433555]]), array([[ 0.20178349],
       [ 0.17787513],
       [-1.44211437]]))

Transalation Vectors:
 (array([[-84.996149 ],
       [ 38.3876432],
       [388.7819472]]), array([[-1.39495647e+02],
       [ 1.61307986e-01],
       [ 3.89458631e+02]]), array([[-34.83454948],
       [  9.43411351],
       [390.63099254]]), array([[-12.51277962],
       [11

In [7]:
img = cv2.imread('myImages/5.png')
h, w = img.shape[:2]
newCameraMatrix, roi = cv2.getOptimalNewCameraMatrix(cameraMatrix, dist, (w,h), 1, (w,h))

print(newCameraMatrix)

with open('intrinsicNew.npy', 'wb') as f:
    np.save(f, newCameraMatrix)

# Undistort
dst = cv2.undistort(img, cameraMatrix, dist, None, newCameraMatrix)

# Crop the Image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imshow('Ditsance', dst)
cv2.waitKey(0)

# Undistort with Remapping
mapx, mapy = cv2.initUndistortRectifyMap(cameraMatrix, dist, None, newCameraMatrix, (w,h), 5)
dst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

# Crop the Image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imshow('Ditsance', dst)
cv2.waitKey(0)

cv2.destroyAllWindows()

[[470.08108521   0.         278.19215297]
 [  0.         520.09613037 233.85179557]
 [  0.           0.           1.        ]]


In [8]:
mean_error = 0

for i in range(len(objPoints)):
    imgPoints2, _ = cv2.projectPoints(objPoints[i], rvecs[i], tvecs[i], cameraMatrix, dist)
    error = cv2.norm(imgPoints[i], imgPoints2, cv2.NORM_L2)/len(imgPoints2)
    mean_error += error
    
print('Total Error: {}'.format(mean_error/len(objPoints)))

Total Error: 0.05300456664517488


In [10]:
cameraMatrix.shape

(3, 3)

In [11]:
dist.shape

(1, 5)